# Feature Analysis
Analyze the processed lap data and feature relationships.

In [4]:
from f1deg.viz.theme import apply_theme

apply_theme()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)

df_clean = pd.read_parquet("../data/processed/laps_clean.parquet")
df_full = pd.read_parquet("../data/processed/laps_full.parquet")
print(f"Clean dataset: {len(df_clean)} laps from {df_clean['race_id'].nunique()} races")
print(f"Full dataset:  {len(df_full)} laps (includes {df_full['is_outlier'].sum()} outliers)")
print(f"Compounds: {sorted(df_clean['compound'].unique())}")
print(f"Circuits:  {df_clean['circuit_id'].nunique()}")
print("\nNew features available:")
print("  traffic_density, position, position_change, race_progress, stint_fraction")
print("  compound_x_track_temp, tyre_life_x_track_temp, is_final_stint")
df_clean.describe()

## Feature Correlations

In [ ]:
numeric_cols = [
    "lap_time_seconds",
    "tyre_life",
    "fuel_mass_kg",
    "air_temp",
    "track_temp",
    "humidity",
    "wind_speed",
    "traffic_density",
    "position",
    "race_progress",
    "stint_fraction",
]
available = [c for c in numeric_cols if c in df_clean.columns]
corr = df_clean[available].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, mask=mask)
ax.set_title("Feature Correlation Matrix (incl. new features)")
plt.tight_layout()
plt.show()

## Fuel Load vs Lap Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_clean["fuel_mass_kg"], df_clean["lap_time_seconds"], alpha=0.1, s=3)
ax.set_xlabel("Fuel Mass (kg)")
ax.set_ylabel("Lap Time (seconds)")
ax.set_title("Fuel Load Effect on Lap Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Compound-Specific Degradation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, compound in zip(axes, ["SOFT", "MEDIUM", "HARD"], strict=False):
    subset = df_clean[df_clean["compound"] == compound]
    color = {"SOFT": "#FF3333", "MEDIUM": "#FFD700", "HARD": "#888888"}[compound]
    ax.scatter(subset["tyre_life"], subset["lap_time_seconds"], alpha=0.1, s=3, c=color)
    # Trend line
    if len(subset) > 10:
        trend = subset.groupby("tyre_life")["lap_time_seconds"].median()
        ax.plot(trend.index, trend.values, "k-", linewidth=2)
    ax.set_xlabel("Tire Life (laps)")
    ax.set_ylabel("Lap Time (s)" if compound == "SOFT" else "")
    ax.set_title(f"{compound}")
    ax.grid(True, alpha=0.3)
plt.suptitle("Degradation by Compound", y=1.02)
plt.tight_layout()
plt.show()

## Per-Circuit Comparison

In [ ]:
pivot = df_clean.groupby(["circuit_id", "compound"])["lap_time_seconds"].mean().unstack()
fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(
    kind="bar",
    ax=ax,
    color=["#FF3333", "#FFD700", "#888888", "#33CC33", "#3399FF"][: len(pivot.columns)],
)
ax.set_ylabel("Mean Lap Time (s)")
ax.set_title("Mean Lap Time by Circuit and Compound")

# Zoom y-axis to ±5s around the data range
y_min = pivot.min().min()
y_max = pivot.max().max()
ax.set_ylim(y_min - 5, y_max + 5)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Traffic Effect on Lap Times

How much does being stuck behind another car slow a driver down?

In [ ]:
from f1deg.viz.theme import COMPOUND as COMPOUND_COLORS

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Traffic density distribution
ax = axes[0]
ax.hist(df_clean["traffic_density"].dropna(), bins=50, alpha=0.7, edgecolor="none")
ax.set_xlabel("Traffic Density")
ax.set_ylabel("Count")
ax.set_title("Traffic Density Distribution")

# 2. Traffic density vs lap time (binned)
ax = axes[1]
df_tmp = df_clean[["traffic_density", "lap_time_seconds", "compound"]].dropna()
bins = pd.qcut(df_tmp["traffic_density"], q=10, duplicates="drop")
grouped = df_tmp.groupby(bins, observed=True)["lap_time_seconds"].agg(["mean", "std"])
ax.errorbar(range(len(grouped)), grouped["mean"], yerr=grouped["std"], fmt="o-", capsize=3)
ax.set_xlabel("Traffic Density Decile")
ax.set_ylabel("Mean Lap Time (s)")
ax.set_title("Lap Time by Traffic Density")

# 3. Traffic effect by compound
ax = axes[2]
for compound in ["SOFT", "MEDIUM", "HARD"]:
    sub = df_clean[df_clean["compound"] == compound]
    if len(sub) < 100:
        continue
    bins = pd.qcut(sub["traffic_density"].dropna(), q=5, duplicates="drop")
    grouped = sub.groupby(bins, observed=True)["lap_time_seconds"].mean()
    ax.plot(
        range(len(grouped)),
        grouped.values,
        "o-",
        label=compound,
        color=COMPOUND_COLORS.get(compound, "#999"),
    )
ax.set_xlabel("Traffic Density Quintile")
ax.set_ylabel("Mean Lap Time (s)")
ax.set_title("Traffic Effect by Compound")
ax.legend()

plt.tight_layout()
plt.show()

## Interaction Features

Do the interaction terms (`compound × track_temp`, `tyre_life × track_temp`) capture non-linear relationships better than the raw features alone?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Tyre life x track temp interaction
ax = axes[0]
temp_bins = pd.cut(
    df_clean["track_temp"].dropna(),
    bins=[0, 25, 35, 45, 60],
    labels=["Cool (<25C)", "Mild (25-35C)", "Hot (35-45C)", "Very Hot (>45C)"],
)
for label in temp_bins.cat.categories:
    mask = temp_bins == label
    sub = df_clean[mask]
    trend = sub.groupby("tyre_life")["lap_time_seconds"].median()
    trend = trend[trend.index <= 40]  # clip for visibility
    ax.plot(trend.index, trend.values, label=label, linewidth=2)
ax.set_xlabel("Tyre Life (laps)")
ax.set_ylabel("Median Lap Time (s)")
ax.set_title("Degradation Rate by Track Temperature")
ax.legend()

# 2. Compound x track temp interaction
ax = axes[1]
for compound in ["SOFT", "MEDIUM", "HARD"]:
    sub = df_clean[df_clean["compound"] == compound]
    if len(sub) < 100:
        continue
    temp_grouped = sub.groupby(pd.cut(sub["track_temp"].dropna(), bins=8))[
        "lap_time_seconds"
    ].mean()
    mid = [interval.mid for interval in temp_grouped.index]
    ax.plot(
        mid,
        temp_grouped.values,
        "o-",
        label=compound,
        color=COMPOUND_COLORS.get(compound, "#999"),
        linewidth=2,
    )
ax.set_xlabel("Track Temperature (C)")
ax.set_ylabel("Mean Lap Time (s)")
ax.set_title("Compound Sensitivity to Track Temperature")
ax.legend()

plt.tight_layout()
plt.show()

## Position & Race Progress Effects

Does track position affect lap times (dirty air, traffic)? How do lap times evolve across a race?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Position vs lap time
ax = axes[0]
pos_grouped = df_clean.groupby("position")["lap_time_seconds"].agg(["mean", "median"])
pos_grouped = pos_grouped[pos_grouped.index <= 20]  # top 20 positions
ax.plot(pos_grouped.index, pos_grouped["median"], "o-", markersize=6)
ax.fill_between(pos_grouped.index, pos_grouped["mean"] - 1, pos_grouped["mean"] + 1, alpha=0.2)
ax.set_xlabel("Track Position")
ax.set_ylabel("Median Lap Time (s)")
ax.set_title("Lap Time by Position (dirty air effect)")

# 2. Race progress vs normalised lap time
ax = axes[1]
df_tmp = df_clean.copy()
race_means = df_tmp.groupby("race_id")["lap_time_seconds"].transform("mean")
df_tmp["norm_lap"] = df_tmp["lap_time_seconds"] / race_means
progress_bins = pd.cut(df_tmp["race_progress"], bins=20)
grouped = df_tmp.groupby(progress_bins, observed=True)["norm_lap"].agg(["mean", "std"])
mid = [interval.mid for interval in grouped.index]
ax.plot(mid, grouped["mean"], "-", linewidth=2)
ax.fill_between(mid, grouped["mean"] - grouped["std"], grouped["mean"] + grouped["std"], alpha=0.2)
ax.set_xlabel("Race Progress (fraction)")
ax.set_ylabel("Normalised Lap Time")
ax.set_title("Pace Evolution Through Race")
ax.axhline(y=1.0, color="#7F849C", linestyle="--", alpha=0.5)

# 3. Position change distribution
ax = axes[2]
pc = df_clean["position_change"].dropna()
ax.hist(pc[pc.between(-5, 5)], bins=np.arange(-5.5, 6.5, 1), alpha=0.7, edgecolor="none")
ax.set_xlabel("Position Change (per lap)")
ax.set_ylabel("Count")
ax.set_title("Position Change Distribution")
ax.axvline(x=0, color="#7F849C", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Outlier & Anomaly Analysis

Examining the laps flagged as outliers or anomalous (from the full dataset).
These are the laps excluded from degradation model training.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Outlier reasons breakdown
ax = axes[0, 0]
if "outlier_reason" in df_full.columns:
    reasons = df_full[df_full["is_outlier"]]["outlier_reason"].value_counts()
    bars = ax.barh(range(len(reasons)), reasons.values, alpha=0.8)
    ax.set_yticks(range(len(reasons)))
    ax.set_yticklabels(reasons.index)
    ax.set_xlabel("Count")
    ax.set_title("Outlier Reasons")
    for bar, val in zip(bars, reasons.values, strict=False):
        ax.text(val + 10, bar.get_y() + bar.get_height() / 2, str(val), va="center")
else:
    ax.text(0.5, 0.5, "No outlier_reason column", ha="center", va="center", transform=ax.transAxes)

# 2. Anomalous laps by race progress
ax = axes[0, 1]
for label, mask, color in [
    ("Normal", ~df_full["is_anomalous_lap"], "#A6E3A1"),
    ("Anomalous", df_full["is_anomalous_lap"], "#F38BA8"),
]:
    sub = df_full[mask]
    ax.hist(sub["race_progress"], bins=30, alpha=0.6, label=label, color=color, density=True)
ax.set_xlabel("Race Progress")
ax.set_ylabel("Density")
ax.set_title("When Do Anomalous Laps Occur?")
ax.legend()

# 3. Retirement distribution across race
ax = axes[1, 0]
retirements = df_full[df_full["did_retire"]].drop_duplicates(subset=["race_id", "driver_id"])
if len(retirements) > 0 and "retirement_lap" in retirements.columns:
    ax.hist(retirements["retirement_lap"].dropna(), bins=30, alpha=0.7, edgecolor="none")
    ax.set_xlabel("Retirement Lap")
    ax.set_ylabel("Count")
    ax.set_title(f"Retirement Timing ({len(retirements)} retirements)")
    ax.axvline(
        x=retirements["retirement_lap"].median(),
        color="#F38BA8",
        linestyle="--",
        label=f"Median: lap {retirements['retirement_lap'].median():.0f}",
    )
    ax.legend()
else:
    ax.text(0.5, 0.5, "No retirement data", ha="center", va="center", transform=ax.transAxes)

# 4. Lap time distribution: normal vs outlier
ax = axes[1, 1]
normal = df_full[~df_full["is_outlier"]]["lap_time_seconds"]
outlier = df_full[df_full["is_outlier"]]["lap_time_seconds"]
ax.hist(normal, bins=100, alpha=0.6, label=f"Normal ({len(normal)})", density=True, color="#A6E3A1")
ax.hist(
    outlier, bins=100, alpha=0.6, label=f"Outlier ({len(outlier)})", density=True, color="#F38BA8"
)
ax.set_xlabel("Lap Time (s)")
ax.set_ylabel("Density")
ax.set_title("Lap Time Distribution: Normal vs Outlier")
ax.legend()
ax.set_xlim(60, 180)

plt.tight_layout()
plt.show()

# Summary stats
print("\n=== Outlier Summary ===")
print(f"Total laps:      {len(df_full)}")
print(f"Outlier laps:    {df_full['is_outlier'].sum()} ({df_full['is_outlier'].mean():.1%})")
print(
    f"Anomalous laps:  {df_full['is_anomalous_lap'].sum()} ({df_full['is_anomalous_lap'].mean():.1%})"
)
print(f"Retirements:     {df_full['did_retire'].sum()} driver-laps from retiring drivers")